# Supply Chain Assistant with Oracle AI Agent Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/agent_memory/02_supply_chain_claude_agent_sdk.ipynb) [![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)


**Framework:** Claude Agent SDK · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook builds a **supply chain assistant** that tracks and updates shipment cargo. The assistant exposes tools to read and mutate shipment state and uses Oracle AI Agent Memory to remember facts about shipments, customers, and operational context across sessions.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational, tool-using | Operations support, customer service, internal ops copilots |
| **Deep Research** | Multi-step autonomous investigation | Literature review, scoping, market research |
| **Workflow** | Predetermined sequence with conditional branches | Approval pipelines, compliance gates, structured triage |

> **📌 This notebook focuses on the Assistant mode.**
>
> An assistant agent is conversational first — it responds to human prompts, calls tools when it needs to read or change state, and carries context across turns. Memory is how it stays coherent across a working day or across sessions with the same operator.

## What You'll Learn

- How to define **in-process tools** with the Claude Agent SDK using the `@tool` decorator
- How to wire those tools into an **MCP server** that the Claude Agent SDK can serve
- How to run the assistant in **multi-turn** mode with `ClaudeSDKClient`
- How to use Oracle AI Agent Memory to persist shipment records, operational notes, and conversation history so the assistant stays coherent across restarts

> **💡 Key insight:** In the Claude Agent SDK, the `@tool` decorator + `create_sdk_mcp_server` pattern is how Python functions become things the LLM can call. You don't define tools inline on the agent; you register them as an MCP server and point the agent at it.

## Prerequisites

- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **`ANTHROPIC_API_KEY`** for the Claude Agent SDK
- **`OPENAI_API_KEY`** for embeddings (OAMP supports OpenAI-compatible embedders)
- The `oracleagentmemory` wheel installed in the active kernel's environment

## 1. Install dependencies

In [1]:
%pip install -q "oracleagentmemory[litellm]" claude-agent-sdk nest_asyncio

Note: you may need to restart the kernel to use updated packages.


## 2. Configure API keys and Oracle connection

In [ ]:
import os

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-")
os.environ.setdefault("OPENAI_API_KEY", "sk-")

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

import nest_asyncio
nest_asyncio.apply()

print("Environment configured.")

Environment configured.


## 3. Connect to Oracle AI Database and initialise the memory client

The assistant uses memory for two things:

1. **Operational facts** — shipment ownership, customer preferences, known issues — stored as durable memories the assistant can recall by semantic search.
2. **Conversation history** — stored as messages on a thread so the assistant picks up where the operator left off.

We use `text-embedding-3-small` for vector search and attach an LLM so the client can automatically extract memories from thread messages.

> **💡 Key insight:** `table_name_prefix` gives each application its own namespace inside a shared Oracle schema. The supply-chain assistant's data won't collide with other OAMP-using apps in the same database.

In [3]:
import oracledb
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database:", connection.version)

extraction_llm = Llm("gpt-4o-mini", temperature=0.2)

memory_client = OracleAgentMemory(
    connection=connection,
    embedder="text-embedding-3-small",
    llm=extraction_llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
    table_name_prefix="supply_",
)
print("Memory client ready.")

Connected to Oracle AI Database: 23.26.0.0.0
Memory client ready.


## 4. Register the operator and the agent

Each human operator is a `user_id`, and the supply-chain assistant itself is an `agent_id`. This lets multiple operators share the same assistant with isolated memory, and lets one operator converse with multiple specialised assistants.

In [4]:
OPERATOR_ID = "operator-morgan"
AGENT_ID = "supply-chain-assistant"

for create_fn, eid, info in [
    (memory_client.add_user, OPERATOR_ID, "Operations manager — West Coast logistics desk."),
    (memory_client.add_agent, AGENT_ID, "Supply chain assistant with read/write tools for shipment state."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

Registered: operator-morgan
Registered: supply-chain-assistant


## 5. Simulate a shipment database

In production, shipment state lives in an ERP or logistics system. For this example we use an in-memory dict. The tools we expose to the agent don't care — they just need read/write methods over *some* shipment store.

> **📌 Production pattern.** Replace `SHIPMENT_DB` with calls to your real system of record (Oracle tables, an ERP API, a microservice). The tool signatures stay the same; only the bodies change.

In [5]:
SHIPMENT_DB = {
    "SHP-1001": {
        "id": "SHP-1001", "origin": "Shanghai", "destination": "Long Beach",
        "status": "in_transit", "eta": "2026-05-02",
        "carrier": "Maersk", "containers": 12, "customer": "Acme Industrial",
    },
    "SHP-1002": {
        "id": "SHP-1002", "origin": "Rotterdam", "destination": "Newark",
        "status": "at_port", "eta": "2026-04-28",
        "carrier": "MSC", "containers": 6, "customer": "Belden Foods",
    },
    "SHP-1003": {
        "id": "SHP-1003", "origin": "Busan", "destination": "Oakland",
        "status": "delayed", "eta": "2026-05-10",
        "carrier": "ONE", "containers": 18, "customer": "Acme Industrial",
    },
}
print(f"Seeded {len(SHIPMENT_DB)} shipments.")

Seeded 3 shipments.


## 6. Define the agent's tools

We define four tools:

| Tool | Read/Write | Purpose |
|---|---|---|
| `list_shipments` | Read | Enumerate all known shipments |
| `get_shipment_status` | Read | Look up the current state of one shipment |
| `update_shipment` | **Write** | Mutate a field on a shipment |
| `recall_operational_notes` | Read | Semantic search over Oracle-backed memory for prior context |

Each is decorated with `@tool(name, description, input_schema)` and returns a structured content dict. The SDK wires the type schema, docstring, and return format into the LLM's tool-call protocol automatically.

> **💡 Key insight:** Return `{"content": [{"type": "text", "text": ...}]}` from every tool. This is the MCP content format the Claude Agent SDK expects. The text content is what the LLM sees on subsequent turns.

In [6]:
import json
from claude_agent_sdk import tool, create_sdk_mcp_server


@tool(
    "list_shipments",
    "List every shipment currently tracked by the system. Returns a JSON array of shipment summaries.",
    {},
)
async def list_shipments(args: dict) -> dict:
    summaries = [
        {"id": s["id"], "status": s["status"], "eta": s["eta"], "customer": s["customer"]}
        for s in SHIPMENT_DB.values()
    ]
    return {"content": [{"type": "text", "text": json.dumps(summaries, indent=2)}]}


@tool(
    "get_shipment_status",
    "Return the full record for one shipment by its ID (e.g. 'SHP-1001').",
    {"shipment_id": str},
)
async def get_shipment_status(args: dict) -> dict:
    sid = args["shipment_id"]
    if sid not in SHIPMENT_DB:
        return {"content": [{"type": "text", "text": f"No shipment with id {sid}."}]}
    return {"content": [{"type": "text", "text": json.dumps(SHIPMENT_DB[sid], indent=2)}]}


@tool(
    "update_shipment",
    (
        "Update one field of one shipment. Use for status transitions, ETA revisions, or "
        "carrier reassignments. Writes the change through to the operational store and "
        "records the update as a durable memory so it survives restarts."
    ),
    {"shipment_id": str, "field": str, "value": str},
)
async def update_shipment(args: dict) -> dict:
    sid, field, value = args["shipment_id"], args["field"], args["value"]
    if sid not in SHIPMENT_DB:
        return {"content": [{"type": "text", "text": f"No shipment with id {sid}."}]}
    if field not in SHIPMENT_DB[sid]:
        return {"content": [{"type": "text", "text": f"Unknown field '{field}' on shipment {sid}."}]}
    before = SHIPMENT_DB[sid][field]
    SHIPMENT_DB[sid][field] = value
    # Record the change as a durable operational memory
    memory_client.add_memory(
        f"Shipment {sid} {field} changed from '{before}' to '{value}'.",
        user_id=OPERATOR_ID, agent_id=AGENT_ID,
        metadata={"shipment_id": sid, "kind": "state_change"},
    )
    return {"content": [{"type": "text",
            "text": f"Updated {sid}.{field}: '{before}' → '{value}'."}]}


@tool(
    "recall_operational_notes",
    (
        "Semantic search over durable operational memory for prior notes relevant to the "
        "query. Use this when the operator references past events, recurring customer "
        "issues, or context that might live in memory rather than the live shipment record."
    ),
    {"query": str, "max_results": int},
)
async def recall_operational_notes(args: dict) -> dict:
    results = memory_client.search(
        args["query"], user_id=OPERATOR_ID, agent_id=AGENT_ID,
        max_results=args.get("max_results", 5),
    )
    if not results:
        return {"content": [{"type": "text", "text": "(no prior notes)"}]}
    lines = [f"- {r.content}  [distance={r.distance:.3f}]" for r in results]
    return {"content": [{"type": "text", "text": "\n".join(lines)}]}


print("Tools defined: list_shipments, get_shipment_status, update_shipment, recall_operational_notes")

Tools defined: list_shipments, get_shipment_status, update_shipment, recall_operational_notes


## 7. Wrap the tools in an MCP server

The Claude Agent SDK doesn't attach tools directly to the agent — it attaches MCP servers. `create_sdk_mcp_server` bundles our four tools into a server the SDK can run in-process.

> **💡 Key insight:** MCP (Model Context Protocol) is how the Claude Agent SDK surfaces tools. The SDK handles the protocol for you; `create_sdk_mcp_server` is the bridge from a list of `@tool`-decorated functions to the MCP surface the agent can see.

In [7]:
supply_server = create_sdk_mcp_server(
    name="supply-chain",
    tools=[list_shipments, get_shipment_status, update_shipment, recall_operational_notes],
)
print("MCP server created: supply-chain")

MCP server created: supply-chain


## 8. Configure the assistant with options

`ClaudeAgentOptions` holds the system prompt, the MCP servers the agent can see, and the tool allowlist. Tool names on the allowlist use the `mcp__<server>__<tool>` convention.

> **📌 Permission mode.** `acceptEdits` auto-approves tool calls — appropriate for a single-operator assistant where the tools are safe. In production you'd likely use `default` and surface tool-use prompts to the operator before mutating state.

In [8]:
from claude_agent_sdk import ClaudeAgentOptions

SYSTEM_PROMPT = """You are a supply chain assistant for the West Coast logistics desk.

Operating rules:
1. When the operator asks about a shipment, use `get_shipment_status` or `list_shipments` first
   rather than speculating from context.
2. Before changing state, confirm the intended change in plain English, then call `update_shipment`.
3. When the operator mentions a customer, carrier, or prior event, call `recall_operational_notes`
   to check for relevant prior context.
4. Be concise. Surface concrete shipment IDs, fields, and values. Avoid restating what the tool
   returned — interpret it for the operator.
"""

options = ClaudeAgentOptions(
    system_prompt=SYSTEM_PROMPT,
    mcp_servers={"sc": supply_server},
    allowed_tools=[
        "mcp__sc__list_shipments",
        "mcp__sc__get_shipment_status",
        "mcp__sc__update_shipment",
        "mcp__sc__recall_operational_notes",
    ],
    permission_mode="acceptEdits",
)
print("Agent options configured.")

Agent options configured.


## 9. Run a multi-turn conversation with `ClaudeSDKClient`

`ClaudeSDKClient` is the multi-turn interface — unlike `query()`, it maintains context across calls. We'll use it as an async context manager and feed a scripted conversation through it, simulating an operator working through their morning shipments.

We also write each turn to an Oracle memory thread so we have a durable log of the conversation for audit, training, or continuity purposes.

> **💡 Key insight:** Use `query()` for one-shot tasks, `ClaudeSDKClient` for ongoing conversations. The client's context window grows with each turn until you close it — that's what makes multi-turn coherence work.

In [9]:
from claude_agent_sdk import ClaudeSDKClient, AssistantMessage, ToolUseBlock, TextBlock
from oracleagentmemory.apis.thread import Message

# Open a thread to log the conversation into OAMP
conv_thread = memory_client.create_thread(user_id=OPERATOR_ID, agent_id=AGENT_ID)

operator_turns = [
    "What shipments are in flight right now?",
    "What's going on with SHP-1003? The customer just asked for an update.",
    "The carrier says SHP-1003 will now arrive on May 14 because of port congestion. Update the ETA.",
    "Remind me who the customer is on SHP-1003, and flag anything you know about them.",
]

async with ClaudeSDKClient(options=options) as client:
    for turn_idx, prompt in enumerate(operator_turns, 1):
        print(f"\n{'=' * 70}\nTurn {turn_idx} — OPERATOR: {prompt}\n{'=' * 70}")
        await client.query(prompt)
        # Log operator turn
        conv_thread.add_messages([Message(role="user", content=prompt)])

        assistant_text_parts = []
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(f"ASSISTANT: {block.text}")
                        assistant_text_parts.append(block.text)
                    elif isinstance(block, ToolUseBlock):
                        print(f"  [tool] {block.name}({block.input})")
        # Log assistant turn
        if assistant_text_parts:
            conv_thread.add_messages([
                Message(role="assistant", content="\n".join(assistant_text_parts)),
            ])


Turn 1 — OPERATOR: What shipments are in flight right now?


/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/oracleagentmemory/core/thread.py:496: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: add_messages_async
  return run_async_in_sync(
/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{"memori...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self._

  [tool] ToolSearch({'query': 'select:mcp__sc__list_shipments', 'max_results': 1})
  [tool] mcp__sc__list_shipments({})
ASSISTANT: Here's what's active right now:

| Shipment | Customer | Status | ETA |
|---|---|---|---|
| SHP-1001 | Acme Industrial | In Transit | May 2 |
| SHP-1002 | Belden Foods | At Port | Apr 28 ⚠️ |
| SHP-1003 | Acme Industrial | Delayed | May 10 ⚠️ |

**Two items need attention:**
- **SHP-1002** has an ETA of Apr 28 — that's already 2 days past today (Apr 30). It's at port but hasn't cleared yet.
- **SHP-1003** is flagged delayed with a May 10 ETA, and it's for the same customer as SHP-1001 (Acme Industrial).

Want me to dig into either of those, or pull operational notes on Acme Industrial or Belden Foods?


/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{\n  "me...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(



Turn 2 — OPERATOR: What's going on with SHP-1003? The customer just asked for an update.
  [tool] ToolSearch({'query': 'select:mcp__sc__get_shipment_status,mcp__sc__recall_operational_notes', 'max_results': 2})
  [tool] mcp__sc__get_shipment_status({'shipment_id': 'SHP-1003'})
  [tool] mcp__sc__recall_operational_notes({'query': 'SHP-1003 Acme Industrial delay', 'max_results': 5})


/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/oracleagentmemory/core/oracleagentmemory.py:698: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: 
  return run_async_in_sync(self._do_search_async, query, scope, max_results, record_types)


ASSISTANT: Here's the full picture on **SHP-1003**:

- **Route:** Busan → Oakland
- **Carrier:** ONE (Ocean Network Express)
- **Containers:** 18
- **Status:** Delayed — ETA pushed to **May 10**

No prior operational notes with a root cause for the delay (the memory hits are just echoes of this session). 

**For the customer (Acme Industrial):** The honest update is that it's delayed to May 10 out of Busan via ONE, no further detail in the record yet. If you want to give them something more substantive, I'd recommend checking directly with ONE for a delay reason — could be port congestion, vessel schedule change, or equipment issues.

Want me to update any fields on this shipment, or do you need to note a reason for the delay once you have it?


/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='The ship...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(



Turn 3 — OPERATOR: The carrier says SHP-1003 will now arrive on May 14 because of port congestion. Update the ETA.
ASSISTANT: To confirm: I'll update **SHP-1003** ETA from May 10 → **May 14**, with port congestion as the reason. Proceeding now.
  [tool] ToolSearch({'query': 'select:mcp__sc__update_shipment', 'max_results': 1})
  [tool] mcp__sc__update_shipment({'shipment_id': 'SHP-1003', 'field': 'eta', 'value': '2026-05-14'})
ASSISTANT: Done. **SHP-1003** ETA is now **May 14** (slipped 4 days from May 10) due to port congestion.

When you update Acme Industrial, worth flagging that **SHP-1001** (their other shipment) is still on track for May 2 via Oakland — so at least part of their order isn't affected. Do you want to note the congestion reason on the record as well?

Turn 4 — OPERATOR: Remind me who the customer is on SHP-1003, and flag anything you know about them.
ASSISTANT: Customer is **Acme Industrial** — let me check for any operational history on them.
  [tool] mcp__sc__rec

## 10. Inspect the memory state

Two things landed in Oracle during the conversation:

1. **Operational memories** written by `update_shipment` (state changes) and by the automatic extractor (from the conversation thread).
2. **Conversation messages** on the thread, which also feed the extractor to create further durable memories.

Let's look at both.

In [10]:
# Durable memories relevant to shipment 1003
hits = memory_client.search(
    "SHP-1003 ETA change", user_id=OPERATOR_ID, agent_id=AGENT_ID, max_results=10,
)
print(f"Durable memories related to SHP-1003: {len(hits)}")
for r in hits:
    print(f"  - {r.content}  [distance={r.distance:.3f}]")

# Raw thread log
messages = conv_thread.get_messages()
print(f"\nConversation messages on thread: {len(messages)}")
for m in messages[:4]:
    print(f"  [{m.role}] {m.content[:100]}...")

# Confirm the live shipment record was mutated too
print(f"\nCurrent SHP-1003 ETA: {SHIPMENT_DB['SHP-1003']['eta']}")

Durable memories related to SHP-1003: 10
  - Shipment SHP-1003 eta changed from '2026-05-10' to '2026-05-14'.  [distance=0.286]
  - Shipment SHP-1003 is delayed with an updated ETA of May 10 due to potential port congestion or vessel schedule changes.  [distance=0.305]
  - What's going on with SHP-1003? The customer just asked for an update.  [distance=0.306]
  - Shipment SHP-1003 for Acme Industrial is Delayed with an ETA of May 10.  [distance=0.320]
  - Shipment SHP-1003 for Acme Industrial is Delayed with an ETA of May 10.  [distance=0.320]
  - The ETA for shipment SHP-1003 for Acme Industrial is now May 14 due to port congestion.  [distance=0.333]
  - The ETA for shipment SHP-1003 for Acme Industrial is now May 14 due to port congestion.  [distance=0.333]
  - Acme Industrial has been informed of the delay for shipment SHP-1003.  [distance=0.344]
  - The carrier says SHP-1003 will now arrive on May 14 because of port congestion. Update the ETA.  [distance=0.356]
  - No specific reas

## 11. Verify continuity — operator returns later

A later session should pick up the operational context without the operator re-explaining. We create a fresh client and ask a question that relies on memory from the earlier turn.

In [11]:
followup = "Earlier we revised an ETA for a shipment headed to Oakland. What's the revised ETA and why did it change?"

print(f"OPERATOR (new session): {followup}\n")

async with ClaudeSDKClient(options=options) as client2:
    await client2.query(followup)
    async for message in client2.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"ASSISTANT: {block.text}")
                elif isinstance(block, ToolUseBlock):
                    print(f"  [tool] {block.name}({block.input})")

OPERATOR (new session): Earlier we revised an ETA for a shipment headed to Oakland. What's the revised ETA and why did it change?

  [tool] ToolSearch({'query': 'select:mcp__sc__list_shipments,mcp__sc__recall_operational_notes', 'max_results': 2})
  [tool] mcp__sc__list_shipments({})
  [tool] mcp__sc__recall_operational_notes({'query': 'revised ETA Oakland shipment', 'max_results': 5})


/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/oracleagentmemory/core/oracleagentmemory.py:698: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: 
  return run_async_in_sync(self._do_search_async, query, scope, max_results, record_types)


ASSISTANT: The shipment in question is **SHP-1003** (Acme Industrial, Busan → Oakland via ONE):

- **Current ETA:** May 14, 2026
- **Previous ETA:** May 10 (itself already a delay from an earlier date)
- **Reason:** Port congestion — the ETA slipped an additional 4 days from May 10 → May 14

Also worth noting: Acme's other shipment, **SHP-1001**, is unaffected and still on track for May 2 via Oakland. Do you want to add a congestion note to SHP-1003's record, or is there anything else to update?


## 12. Cleanup

In [12]:
# Uncomment to delete the conversation thread and its messages
# memory_client.delete_thread(conv_thread.thread_id)

connection.close()
print("Connection closed.")

Connection closed.


## Key Takeaways

> **🎯 1. `@tool` + `create_sdk_mcp_server` is the Claude Agent SDK's tool-definition pattern.** You don't pass bare Python functions to the agent — you wrap them in an MCP server and point the agent's options at it. The indirection pays off when you scale to multiple tool bundles.

> **🎯 2. Tools should return MCP content blocks.** `{"content": [{"type": "text", "text": ...}]}` is the canonical shape. The LLM sees whatever is in `text` on subsequent turns, so return concise, structured information — not raw dumps.

> **🎯 3. Assistants need two memory tiers too.** Short-term context comes from the multi-turn client's internal history. Durable operational context ("this customer had three delays last quarter") needs explicit writes through a memory tool plus automatic extraction from the thread.

> **🎯 4. Every write that matters should hit memory, not just the live store.** The `update_shipment` tool mutates the live shipment record *and* writes a memory record describing the change. The first is the system of record; the second is how the assistant stays coherent with the operator about what changed and why.

> **🎯 5. Oracle AI Database unifies the substrate.** The live shipment store, the durable memories, and the conversation thread can all live in the same database — one connection pool, one backup story, one compliance review. For enterprise logistics systems that is a material operational win.